In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

In [26]:
ref_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
ref_model.set_use_module_grafting(False)
ref_model.set_use_scc_status(False)

In [28]:
outputs = ref_model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.6582, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [29]:
ref_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Load Trained Weights

In [30]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [31]:
from train_eval_func import eval_loop

In [32]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [33]:
ref_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [34]:
eval_loop(ref_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

# 5. Helper Functions

In [35]:
from layer_merge_helper import reorder_merged_layer_heads, reorder_merged_layer_neurons

In [36]:
from structure_prune_helper import compute_importance_scores, register_importance_masks, remove_importance_masks

In [37]:
register_importance_masks(model, device, register_heads=True, register_ffn=True)

  Layer 0: Registered masks (12 heads, 3072 neurons)
  Layer 1: Registered masks (12 heads, 3072 neurons)
  Layer 2: Registered masks (12 heads, 3072 neurons)
  Layer 3: Registered masks (12 heads, 3072 neurons)
  Layer 4: Registered masks (12 heads, 3072 neurons)
  Layer 5: Registered masks (12 heads, 3072 neurons)
  Layer 6: Registered masks (12 heads, 3072 neurons)
  Layer 7: Registered masks (12 heads, 3072 neurons)
  Layer 8: Registered masks (12 heads, 3072 neurons)
  Layer 9: Registered masks (12 heads, 3072 neurons)
  Layer 10: Registered masks (12 heads, 3072 neurons)
  Layer 11: Registered masks (12 heads, 3072 neurons)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [38]:
head_importance, neuron_importance = compute_importance_scores(model, train_dataloader, device, compute_heads=True, compute_ffn=True)

Computing importance (heads + FFN): 100%|██████████| 115/115 [00:13<00:00,  8.48it/s]


In [39]:
from train_eval_func import get_primary_metric
import os
from layer_merge_helper import merge_mha, merge_ff

In [40]:
from torch.optim import AdamW
from train_eval_func import train

In [41]:
def iternative_top_k_merge_and_train(
    model,
    train_dataloader,
    val_dataloader,
    task_name,
    device,
    cka_evaluator,
    init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],
    recovery_epochs=3,
    recovery_lr=1e-5,
    patience=2,
    save_dir='./weights/',
    keep_temp_checkpoints=False,
    cka_max_iter=float("Inf"),
    teacher_model=None,
    alpha=0.5,
    temperature=6
):
    orig_n_layer = len(model.bert.encoder.layer)
    target_metric = get_primary_metric(task_name)
    threshold = init_metric[target_metric] * 0.01

    # Create recovery checkpoint directory
    recovery_dir = os.path.join(save_dir, 'recovery')
    os.makedirs(recovery_dir, exist_ok=True)
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"Target Metric for {task_name}: {target_metric}")
    print(f"Original {target_metric} score: {init_metric[target_metric]:.4f}")
    print(f"Recovery threshold: {threshold:.4f}")
    print("")

    for merge_itr in range(num_merges):
        print("=" * 70)
        print(f"Merge Iteration: {merge_itr + 1}/{num_merges}")
        print("=" * 70)

        # Layer Similarity Estimation
        print("\n[1/5] Computing CKA Similarity...")
        cls_reps_similarity = cka_evaluator.pairwise(
            model, 
            train_dataloader, 
            device, 
            only_cls_token=True, 
            max_iter=cka_max_iter
        )
        stats = cka_evaluator.similarity_stats(cls_reps_similarity)
        
        print("  Similarity Stats:")
        print(f"    Average: {stats['average']:.6f}")
        print(f"    Highest: {stats['max']:.6f}")
        print(f"    Lowest:  {stats['min']:.6f}")

        # Merge Most Adjacent Similar Layer Pair
        print("\n[2/5] Merging Layers...")
        merge_layer = stats['argmax']
        print("Merge Layer:", merge_layer)

        head_importance, neuron_importance = compute_importance_scores(model, train_dataloader, device, compute_heads=True, compute_ffn=True)

        head_imp1, indices1 = torch.sort(head_importance[merge_layer], descending=True)
        head_imp2, indices2 = torch.sort(head_importance[merge_layer+1], descending=True)

        neuron_imp1, indices1 = torch.sort(neuron_importance[merge_layer], descending=True)        
        neuron_imp2, indices1 = torch.sort(neuron_importance[merge_layer+1], descending=True)
        
        merge_mha(model, model.bert.encoder.layer[merge_layer], model.bert.encoder.layer[merge_layer+1], head_imp1, head_imp2, device, 4)
        merge_ff(model, model.bert.encoder.layer[merge_layer], model.bert.encoder.layer[merge_layer+1], neuron_imp1, neuron_imp2, device, extra_neurons=700)

        del model.bert.encoder.layer[merge_layer]

        print("\n[3/5] Evaluating Post-Merge Performance...")
        eval_metric = eval_loop(model, val_dataloader, task_name, device)[0]
        print("  Metrics:")
        for metric_name, value in eval_metric.items():
            marker = "★" if metric_name == target_metric else " "
            print(f"    {marker} {metric_name}: {value:.4f}")

        # Recovery Training
        print("\n[4/5] Recovery Training...")
        diff = init_metric[target_metric] - eval_metric[target_metric]
        print(f"  Performance drop: {diff:.4f} (threshold: {threshold:.4f})")

        is_retrained = False
        if diff > threshold:
            print("  → Recovery training NEEDED")
            
            # Setup optimizer and scheduler
            optimizer = AdamW(model.parameters(), lr=recovery_lr)
            num_training_steps = recovery_epochs * len(train_dataloader)
            # lr_scheduler = set_lr_scheduler(
            #     optimizer=optimizer,
            #     num_training_steps=num_training_steps
            # )
            
            print(f"  → Training: {recovery_epochs} epochs, lr={recovery_lr}")
            
            # Temporary checkpoint path
            temp_save_path = os.path.join(recovery_dir, f'temp_iter{merge_itr}.pt')
            
            # Train with checkpoint saving
            lr_scheduler= None
            train_stats = train(
                model,
                train_dataloader,
                val_dataloader,
                task_name,
                device,
                num_epochs=recovery_epochs,
                lr=recovery_lr,
                patience=patience,
                use_early_stopping=True,
                optimizer=optimizer,
                lr_scheduler=lr_scheduler,
                save_path=temp_save_path,
                display_epoch_iter=True,
                teacher_model=teacher_model, alpha=alpha, temperature=temperature
            )

             # CRITICAL: Reload best checkpoint
            print("\n  → Loading best checkpoint...")
            best_checkpoint = torch.load(temp_save_path)
            model.load_state_dict(best_checkpoint['model_state_dict'])
            
            print(f"  ✓ Loaded best checkpoint from epoch {best_checkpoint['epoch']}")
            print(f"    Best val loss: {best_checkpoint['val_loss']:.4f}")
            print(f"    Best val metrics: {best_checkpoint['val_metrics']}")
            
            # merge_history['metrics_after_recovery'].append(best_checkpoint['val_metrics'])
            # merge_history['training_stats'].append(train_stats)
            
            # Clean up temp checkpoint if not keeping
            if not keep_temp_checkpoints:
                os.remove(temp_save_path)
                print(f"  ✓ Cleaned up temporary checkpoint")

            is_retrained = True
        else:
            print("  → Recovery training NOT needed")
            # merge_history['metrics_after_recovery'].append(eval_metric)
            # merge_history['training_stats'].append(None)
        
        # Save final model at target layer counts
        print("\n[5/5] Saving Model...")
        n_layer = orig_n_layer - merge_itr - 1
        if n_layer in target_layers:
            save_path = os.path.join(save_dir, f'layer-{n_layer}-{task_name}.pt')
            
            # save_object = {
            #     'model_state_dict': model.state_dict(),
            #     'layer_track': tracker.get_mapping(),
            #     'train_stats': merge_history['training_stats'][-1],
            #     'metrics_after_merge': merge_history['metrics_after_merge'][-1],
            #     'metrics_after_recovery': merge_history['metrics_after_recovery'][-1],
            #     'merge_iteration': merge_itr + 1,
            #     'num_layers': n_layer,
            #     'is_retrained': is_retrained
            # }
            
            # torch.save(save_object, save_path)
            print(f"  ✓ Saved {n_layer}-layer model to {save_path}")
        else:
            print(f"  • {n_layer} layers (not in target list, skipping save)")
        
        # # Store iteration info
        # merge_history['iterations'].append({
        #     'iteration': merge_itr + 1,
        #     'merged_layers': (merge_layer, merge_layer + 1),
        #     'num_layers_after': n_layer
        # })
        # merge_history['layer_compositions'].append(tracker.get_mapping())
        
        print("")
    
    # Final summary
    print("=" * 70)
    print("MERGE LOOP COMPLETED")
    print("=" * 70)
    print(f"Total merges: {num_merges}")
    # print(f"Final layers: {len(tracker)}")
    # print(f"Final composition: {tracker.get_mapping()}")

    torch.cuda.empty_cache()

In [42]:
init_metric = eval_loop(model, val_dataloader, task_name, device)[0]
init_metric

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [43]:
from CKA import CKAEvaluator

In [44]:
cka_eval = CKAEvaluator(device)

In [45]:
iternative_top_k_merge_and_train(
    model, train_dataloader, val_dataloader, task_name, device, cka_evaluator=cka_eval, init_metric=init_metric, num_merges=6
)

Target Metric for mrpc: f1
Original f1 score: 0.9094
Recovery threshold: 0.0091

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 115/115 [00:19<00:00,  5.79it/s]


  Similarity Stats:
    Average: 0.899457
    Highest: 0.993674
    Lowest:  0.724031

[2/5] Merging Layers...
Merge Layer: 9


Computing importance (heads + FFN): 100%|██████████| 115/115 [00:14<00:00,  8.02it/s]


9 7
2237 1535

[3/5] Evaluating Post-Merge Performance...
  Metrics:
      accuracy: 0.8750
    ★ f1: 0.9122

[4/5] Recovery Training...
  Performance drop: -0.0028 (threshold: 0.0091)
  → Recovery training NOT needed

[5/5] Saving Model...
  • 11 layers (not in target list, skipping save)

Merge Iteration: 2/6

[1/5] Computing CKA Similarity...


...

Training:  66%|██████▋   | 229/345 [00:30<00:11,  9.79it/s]

✓ Saved checkpoint (best loss: 1.1718)

<----------------- Epoch 2 ----------------->
Loss: 0.08, Training Metrics:
  accuracy: 0.9700
  f1: 0.9777
Validation Loss: 1.17, Validation Metrics:
  accuracy: 0.8064
  f1: 0.8728
Elapsed Time: 11.3415 sec


Training: 100%|██████████| 345/345 [01:00<00:00,  5.70it/s]

✓ Saved checkpoint (best loss: 1.1476)

<----------------- Epoch 3 ----------------->
Loss: 0.07, Training Metrics:
  accuracy: 0.9760
  f1: 0.9823
Validation Loss: 1.15, Validation Metrics:
  accuracy: 0.8260
  f1: 0.8795
Elapsed Time: 11.4231 sec

Total Training Time: 60.52 sec
Chosen Primary Metric: f1
Best Metric Run: 0.8795

  → Loading best checkpoint...


  ✓ Loaded best checkpoint from epoch 3
    Best val loss: 1.1476
    Best val metrics: {'accuracy': 0.8259803921568627, 'f1': 0.8794567062818336}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./weights/layer-6-mrpc.pt

MERGE LOOP COMPLETED
Total merges: 6


In [39]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [45]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8774509803921569, 'f1': 0.9140893470790378}

In [46]:
eval_loop(ref_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [50]:
len(model.bert.encoder.layer)

11

In [55]:
model.bert.encoder.layer[9]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=1024, bias=True)
      (key): Linear(in_features=768, out_features=1024, bias=True)
      (value): Linear(in_features=768, out_features=1024, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=1024, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3772, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3772, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [51]:
len(ref_model.bert.encoder.layer)

12

In [56]:
def test_mha_merge(
    model,              # merged model
    ref_model,          # clone BEFORE merge
    source_layer_idx,
    target_layer_idx,
    s_idx, t_idx,       # number of heads taken
):
    layer_m = model.bert.encoder.layer[target_layer_idx - 1]
    layer_s = ref_model.bert.encoder.layer[source_layer_idx]
    layer_t = ref_model.bert.encoder.layer[target_layer_idx]

    head_size = layer_m.attention.self.attention_head_size

    # Convert heads → rows
    s_len = s_idx * head_size
    t_len = t_idx * head_size

    print(f"\nTesting MHA Merge:")
    print(f"Target heads kept: {t_idx}, Source heads copied: {s_idx}")

    # ---------------------------
    # 1. TEST TARGET PART
    # ---------------------------
    print("\n[1] Checking TARGET part...")

    for h in range(t_idx):
        start = h * head_size
        end   = (h + 1) * head_size

        ok = torch.allclose(
            layer_m.attention.self.query.weight[start:end],
            layer_t.attention.self.query.weight[start:end]
        )

        print(f"  Head {h} (target) match: {ok}")

    # ---------------------------
    # 2. TEST SOURCE PART
    # ---------------------------
    print("\n[2] Checking SOURCE part...")

    for h in range(s_idx):
        src_start = h * head_size
        src_end   = (h + 1) * head_size

        tgt_start = t_len + h * head_size
        tgt_end   = tgt_start + head_size

        ok = torch.allclose(
            layer_m.attention.self.query.weight[tgt_start:tgt_end],
            layer_s.attention.self.query.weight[src_start:src_end]
        )

        print(f"  Head {h} (source → merged) match: {ok}")

    # ---------------------------
    # 3. FINAL CHECK
    # ---------------------------
    print("\n[3] Summary:")
    print("✔ If all True → merge is correct")

In [57]:
test_mha_merge(
    model,
    ref_model,
    source_layer_idx=9,
    target_layer_idx=10,
    s_idx=9,
    t_idx=7
)


Testing MHA Merge:
Target heads kept: 7, Source heads copied: 9

[1] Checking TARGET part...
  Head 0 (target) match: False
  Head 1 (target) match: False
  Head 2 (target) match: False
  Head 3 (target) match: False
  Head 4 (target) match: False
  Head 5 (target) match: False
  Head 6 (target) match: False

[2] Checking SOURCE part...
  Head 0 (source → merged) match: True
  Head 1 (source → merged) match: True
  Head 2 (source → merged) match: True
  Head 3 (source → merged) match: True
  Head 4 (source → merged) match: True
  Head 5 (source → merged) match: True
  Head 6 (source → merged) match: True
  Head 7 (source → merged) match: True
  Head 8 (source → merged) match: True

[3] Summary:
✔ If all True → merge is correct


In [40]:
for idx, layer in enumerate(model.bert.encoder.layer):
    reorder_merged_layer_heads(layer, head_importance[idx], device)
    reorder_merged_layer_neurons(layer, neuron_importance[idx], device)

In [41]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [42]:
for idx, layer in enumerate(ref_model.bert.encoder.layer):
    reorder_merged_layer_heads(layer, head_importance[idx], device)
    reorder_merged_layer_neurons(layer, neuron_importance[idx], device)

In [43]:
eval_loop(ref_model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [39]:
from layer_merge_helper import merge_mha, merge_ff

In [45]:
head_imp1, indices1 = torch.sort(head_importance[10], descending=True)

In [46]:
head_imp2, indices2 = torch.sort(head_importance[11], descending=True)

In [47]:
neuron_imp1, indices1 = torch.sort(neuron_importance[10], descending=True)

In [48]:
neuron_imp2, indices1 = torch.sort(neuron_importance[11], descending=True)

In [49]:
merge_mha(model, model.bert.encoder.layer[10], model.bert.encoder.layer[11], head_imp1, head_imp2, device, 4)

9 7


In [52]:
def test_mha_merge(
    model,              # merged model
    ref_model,          # clone BEFORE merge
    source_layer_idx,
    target_layer_idx,
    s_idx, t_idx,       # number of heads taken
):
    layer_m = model.bert.encoder.layer[target_layer_idx]
    layer_s = ref_model.bert.encoder.layer[source_layer_idx]
    layer_t = ref_model.bert.encoder.layer[target_layer_idx]

    head_size = layer_m.attention.self.attention_head_size

    # Convert heads → rows
    s_len = s_idx * head_size
    t_len = t_idx * head_size

    print(f"\nTesting MHA Merge:")
    print(f"Target heads kept: {t_idx}, Source heads copied: {s_idx}")

    # ---------------------------
    # 1. TEST TARGET PART
    # ---------------------------
    print("\n[1] Checking TARGET part...")

    for h in range(t_idx):
        start = h * head_size
        end   = (h + 1) * head_size

        ok = torch.allclose(
            layer_m.attention.self.query.weight[start:end],
            layer_t.attention.self.query.weight[start:end]
        )

        print(f"  Head {h} (target) match: {ok}")

    # ---------------------------
    # 2. TEST SOURCE PART
    # ---------------------------
    print("\n[2] Checking SOURCE part...")

    for h in range(s_idx):
        src_start = h * head_size
        src_end   = (h + 1) * head_size

        tgt_start = t_len + h * head_size
        tgt_end   = tgt_start + head_size

        ok = torch.allclose(
            layer_m.attention.self.query.weight[tgt_start:tgt_end],
            layer_s.attention.self.query.weight[src_start:src_end]
        )

        print(f"  Head {h} (source → merged) match: {ok}")

    # ---------------------------
    # 3. FINAL CHECK
    # ---------------------------
    print("\n[3] Summary:")
    print("✔ If all True → merge is correct")

In [51]:
test_mha_merge(
    model,
    ref_model,
    source_layer_idx=10,
    target_layer_idx=11,
    s_idx=9,
    t_idx=7
)


Testing MHA Merge:
Target heads kept: 7, Source heads copied: 9

[1] Checking TARGET part...
  Head 0 (target) match: False
  Head 1 (target) match: False
  Head 2 (target) match: False
  Head 3 (target) match: False
  Head 4 (target) match: False
  Head 5 (target) match: False
  Head 6 (target) match: False

[2] Checking SOURCE part...
  Head 0 (source → merged) match: True
  Head 1 (source → merged) match: True
  Head 2 (source → merged) match: True
  Head 3 (source → merged) match: True
  Head 4 (source → merged) match: True
  Head 5 (source → merged) match: True
  Head 6 (source → merged) match: True
  Head 7 (source → merged) match: True
  Head 8 (source → merged) match: True

[3] Summary:
✔ If all True → merge is correct


In [52]:
model.bert.encoder.layer[3].attention.self.query.weight.shape

torch.Size([768, 768])

In [53]:
merge_ff(model, model.bert.encoder.layer[10], model.bert.encoder.layer[11], neuron_imp1, neuron_imp2, device, extra_neurons=700)

2423 1349


In [54]:
def test_ffn_merge(
    model,              # merged model
    ref_model,          # clone BEFORE merge
    source_layer_idx,
    target_layer_idx,
    s_idx, t_idx
):
    layer_m = model.bert.encoder.layer[target_layer_idx]
    layer_s = ref_model.bert.encoder.layer[source_layer_idx]
    layer_t = ref_model.bert.encoder.layer[target_layer_idx]

    s_len = s_idx
    t_len = t_idx

    print(f"\nTesting FFN Merge:")
    print(f"Target neurons kept: {t_idx}, Source neurons copied: {s_idx}")

    # ---------------------------
    # 1. INTERMEDIATE LAYER (row-wise)
    # ---------------------------
    print("\n[1] Checking INTERMEDIATE (dense) layer...")

    # Target part
    ok_target = torch.allclose(
        layer_m.intermediate.dense.weight[:t_len],
        layer_t.intermediate.dense.weight[:t_len]
    )
    print(f"  Target part match: {ok_target}")

    # Source part
    ok_source = torch.allclose(
        layer_m.intermediate.dense.weight[t_len:t_len+s_len],
        layer_s.intermediate.dense.weight[:s_len]
    )
    print(f"  Source part match: {ok_source}")

    # ---------------------------
    # 2. OUTPUT LAYER (column-wise)
    # ---------------------------
    print("\n[2] Checking OUTPUT (dense) layer...")

    # Target part
    ok_target_out = torch.allclose(
        layer_m.output.dense.weight[:, :t_len],
        layer_t.output.dense.weight[:, :t_len]
    )
    print(f"  Target part match: {ok_target_out}")

    # Source part
    ok_source_out = torch.allclose(
        layer_m.output.dense.weight[:, t_len:t_len+s_len],
        layer_s.output.dense.weight[:, :s_len]
    )
    print(f"  Source part match: {ok_source_out}")

    # ---------------------------
    # 3. BIAS CHECK (intermediate only)
    # ---------------------------
    print("\n[3] Checking INTERMEDIATE bias...")

    ok_bias_target = torch.allclose(
        layer_m.intermediate.dense.bias[:t_len],
        layer_t.intermediate.dense.bias[:t_len]
    )
    print(f"  Target bias match: {ok_bias_target}")

    ok_bias_source = torch.allclose(
        layer_m.intermediate.dense.bias[t_len:t_len+s_len],
        layer_s.intermediate.dense.bias[:s_len]
    )
    print(f"  Source bias match: {ok_bias_source}")

    # ---------------------------
    # FINAL SUMMARY
    # ---------------------------
    print("\n[4] Summary:")
    print("✔ All True → FFN merge is correct")

In [55]:
test_ffn_merge(
    model,
    ref_model,
    source_layer_idx=10,
    target_layer_idx=11,
    s_idx=2423,
    t_idx=1249
)


Testing FFN Merge:
Target neurons kept: 1249, Source neurons copied: 2423

[1] Checking INTERMEDIATE (dense) layer...
  Target part match: False
  Source part match: False

[2] Checking OUTPUT (dense) layer...
  Target part match: False
  Source part match: False

[3] Checking INTERMEDIATE bias...
  Target bias match: False
  Source bias match: False

[4] Summary:
✔ All True → FFN merge is correct


In [56]:
remove_importance_masks(model)

✓ Removed head and FFN masks


In [57]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.875, 'f1': 0.9142857142857143}